# Cleanup, close, and kill Alibre Design

Every way to tear down state: the document level, the
application level, and the OS level.

>  **Caution:** most cells here are *destructive*. They close
> documents or terminate Alibre. Each destructive cell is **commented
> out** so a "Run All" doesn't wipe out your session. Uncomment the
> lines you want to execute.

## What's automatic at script exit?

**Typical scripts need no explicit teardown.** When your
Python process exits:

1. PythonNet's CLR shutdown runs finalizers on the COM wrappers,
   which `Marshal.Release`s the underlying `__ComObject`. Alibre's
   `AutomationHook` refcount drops by one.
2. **Alibre itself keeps running.** The hook is a singleton owned
   by Alibre's process. Disconnecting your script doesn't close
   Alibre or any open documents.
3. Normal scripts require no `try/finally` cleanup.

Cases where you *do* want explicit action:

| Scenario | What to do |
|---|---|
| Long-running Jupyter kernel + Alibre restarted out-of-band | `reset_alibre()`: drops the stale cached proxy |
| You subscribed to events (`em.OnSessionOpen += handler`) | unsubscribe with `em.OnSessionOpen -= handler` before exit |
| Need Alibre itself to exit when your script does | `root.Terminate()` before exiting (see section 5) |

## Inspect the state first (safe)

In [ ]:
from alibrex import connect
root = connect()
root.Sessions.Count

In [ ]:
for i in range(root.Sessions.Count):
    s = root.Sessions.Item(i)
    print(f"  [{i}] {s.Name:30s}  type={s.SessionType}")

## 1. Save the active document

Non-destructive: writes the document to its current path.

In [ ]:
from alibrex import CurrentPart
part = CurrentPart()
# part.Save()                          # uncomment to save in place

## 2. Save As: write to a new path

`SaveAs(destination, itemName)` writes the document to a new file
and switches the session to that path.

In [ ]:
# part.SaveAs(r"C:\path\to\NewName.AD_PRT", "NewName")

## 3. Close the active document

`Close(saveSession: bool)`: pass `True` to save before closing,
`False` to discard unsaved changes.

In [ ]:
# part.Close(True)                     # save, then close
# part.Close(False)                    # discard changes, close

## 4. Close every open document

Walk `root.Sessions` backwards so the index stays valid as items
are removed.

In [ ]:
# for i in range(root.Sessions.Count - 1, -1, -1):
#     root.Sessions.Item(i).Close(False)   # False = discard
# root.Sessions.Count

## 5. Terminate Alibre Design (graceful, via API)

`IADRoot.Terminate()` shuts the Alibre process down cleanly. After
this the bound proxy is dead. Call `reset_alibre()` so the next
`CurrentPart()` rebinds to a fresh instance.

In [ ]:
# root.Terminate()
# from alibrex import reset_alibre
# reset_alibre()

## 6. `TerminateAll()`: close every Alibre instance

Identical to `Terminate()` for single-process installs. On setups
with multiple Alibre processes, this closes them all.

In [ ]:
# root.TerminateAll()

## 7. Drop the Python-side cache only (safe)

`reset_alibre()` doesn't touch Alibre; it forgets the cached
proxy on the Python side. Useful when you've manually closed and
reopened Alibre and want the next `CurrentPart()` to rebind.

In [ ]:
from alibrex import reset_alibre
reset_alibre()

## 8. OS-level kill (last resort)

If Alibre is hung and `Terminate()` doesn't respond, kill the
process from Windows. **Forceful.** Unsaved changes are lost
and the COM hook may need a moment to clear.

In [ ]:
import subprocess
# subprocess.run(["taskkill", "/F", "/IM", "Alibre Design.exe"])

## Summary

| Action | Method | Saves? | Destructive? |
|---|---|---|---|
| Save active doc | `session.Save()` | yes | no |
| Save as new path | `session.SaveAs(path, name)` | yes | no |
| Close active doc | `session.Close(save: bool)` | optional | yes |
| Close all docs | iterate `root.Sessions` + `Close(False)` | no | yes |
| Quit Alibre | `root.Terminate()` | no | yes |
| Quit all Alibres | `root.TerminateAll()` | no | yes |
| Drop Python cache | `reset_alibre()` | n/a | no |
| Force-kill process | `taskkill /F /IM "Alibre Design.exe"` | no | yes |